# Part 3 — Training-Time Optimisation Techniques

Five techniques, each with a short explanation, an example snippet, and a
**controlled experiment** on the *same* model, data, batch size and number of
steps &mdash; measuring **time**, **GPU memory** (where available) and **loss**.

1. Tensor creation (CPU vs GPU)
2. Weight initialisation
3. Activation checkpointing
4. Gradient accumulation
5. Mixed-precision training

> **Run in Colab with a GPU runtime** (Runtime &rarr; Change runtime type &rarr; GPU)
> so the memory columns are populated. On CPU the code still runs; GPU-only
> measurements show as `None`.


## 0. Shared setup (one model / data / batch / steps for everything)

In [1]:
# --- Local GPU note ---
# This part needs the CUDA build of PyTorch to use your RTX 3060.
#   pip install torch --index-url https://download.pytorch.org/whl/cu124
import torch, torch.nn as nn, time
import numpy as np, pandas as pd
from torch.utils.checkpoint import checkpoint_sequential

torch.manual_seed(0); np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_CUDA = torch.cuda.is_available()
print("Device:", device, "| CUDA:", USE_CUDA)
if USE_CUDA:
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: running on CPU -> memory columns will be None and AMP/"
          "checkpointing show little benefit. Install the CUDA torch build.")

# ---- fixed config used by every experiment ----
# Sized so ACTIVATIONS dominate memory (big batch, several wide layers) -> the
# memory-saving techniques (checkpointing / accumulation / AMP) actually show up.
IN_DIM, HID, OUT, DEPTH = 1024, 2048, 10, 8
BATCH, STEPS, LR = 8192, 20, 1e-2
# If you hit CUDA out-of-memory on a 6 GB card, shrink: BATCH=4096 (or HID=1024).
# Relative comparisons between techniques stay valid at the smaller size.

N = BATCH * STEPS
X = torch.randn(N, IN_DIM)
W_teacher = torch.randn(IN_DIM, OUT)   # fixed teacher -> makes the task LEARNABLE
Y = (X @ W_teacher).argmax(1)          # so final_loss is a meaningful measure

def make_model(init="default"):
    layers, d = [], IN_DIM
    for _ in range(DEPTH):
        lin = nn.Linear(d, HID)
        if init == "kaiming":
            nn.init.kaiming_normal_(lin.weight, nonlinearity="relu"); nn.init.zeros_(lin.bias)
        elif init == "xavier":
            nn.init.xavier_normal_(lin.weight); nn.init.zeros_(lin.bias)
        layers += [lin, nn.ReLU()]
        d = HID
    layers += [nn.Linear(d, OUT)]
    return nn.Sequential(*layers)

results = []   # collected experiment rows -> summary table at the end


Device: cuda | CUDA: True
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


### A reusable training loop

`accum` = gradient-accumulation steps, `checkpoint_seg` = # activation-checkpoint
segments (0 = off), `use_amp` = mixed precision. Everything else is fixed.

In [2]:
def forward(model, x, seg):
    if seg and model.training:
        return checkpoint_sequential(model, seg, x, use_reentrant=False)
    return model(x)

def run_training(model, label, use_amp=False, accum=1, checkpoint_seg=0):
    model = model.to(device).train()
    opt = torch.optim.SGD(model.parameters(), lr=LR)
    lossfn = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp and USE_CUDA)  # new API
    amp_dev = "cuda" if USE_CUDA else "cpu"

    if USE_CUDA:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
    t0 = time.perf_counter()

    micro = BATCH // accum
    last = None
    for s in range(STEPS):
        i = (s * BATCH) % N
        xb, yb = X[i:i+BATCH].to(device), Y[i:i+BATCH].to(device)
        opt.zero_grad(set_to_none=True)
        running = 0.0
        for a in range(accum):
            xs, ys = xb[a*micro:(a+1)*micro], yb[a*micro:(a+1)*micro]
            with torch.autocast(device_type=amp_dev, enabled=use_amp):
                loss = lossfn(forward(model, xs, checkpoint_seg), ys) / accum
            scaler.scale(loss).backward()
            running += loss.item() * accum
        scaler.step(opt); scaler.update()
        last = running / accum

    if USE_CUDA: torch.cuda.synchronize()
    dt = time.perf_counter() - t0
    mem = round(torch.cuda.max_memory_allocated()/1e6, 1) if USE_CUDA else None
    row = {"experiment": label, "time_s": round(dt, 3),
           "peak_mem_MB": mem, "final_loss": round(last, 4)}
    results.append(row); print(row)
    return row


## 1. Tensor creation: CPU vs GPU

Creating a tensor on the GPU (or moving one there) lets subsequent math run on
the device. GPUs win big on large parallel ops (matmul); for tiny tensors the
transfer overhead can make CPU faster.

In [3]:
def time_matmul(dev, n=4096, iters=10):
    a = torch.randn(n, n, device=dev); b = torch.randn(n, n, device=dev)
    if dev == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters):
        c = a @ b
    if dev == "cuda": torch.cuda.synchronize()
    return round((time.perf_counter() - t0) / iters * 1000, 2)  # ms/iter

cpu_ms = time_matmul("cpu")
print(f"CPU  matmul: {cpu_ms} ms/iter")
if USE_CUDA:
    gpu_ms = time_matmul("cuda")
    print(f"GPU  matmul: {gpu_ms} ms/iter  ->  ~{cpu_ms/gpu_ms:.1f}x faster")
else:
    print("No GPU available -> run on a Colab GPU runtime to see the speedup.")

# example snippet: explicit device placement
t_cpu = torch.ones(3, 3)                       # created on CPU
t_gpu = torch.ones(3, 3, device=device)        # created directly on device
moved = t_cpu.to(device)                        # moved CPU -> device
print("devices:", t_cpu.device, t_gpu.device, moved.device)


CPU  matmul: 622.21 ms/iter
GPU  matmul: 45.16 ms/iter  ->  ~13.8x faster
devices: cpu cuda:0 cuda:0


## 2. Weight initialisation

Good init keeps activation/gradient variance stable, so training starts from a
lower loss and converges faster. We compare PyTorch default vs Kaiming vs Xavier
over the same `STEPS`.

In [4]:
for init in ["default", "kaiming", "xavier"]:
    run_training(make_model(init), label=f"init={init}")


{'experiment': 'init=default', 'time_s': 6.93, 'peak_mem_MB': 847.8, 'final_loss': 2.3029}
{'experiment': 'init=kaiming', 'time_s': 6.846, 'peak_mem_MB': 847.8, 'final_loss': 2.3139}
{'experiment': 'init=xavier', 'time_s': 6.814, 'peak_mem_MB': 847.8, 'final_loss': 2.303}


## 3. Activation checkpointing

Instead of storing every layer's activations for the backward pass, checkpointing
**recomputes** them during backprop &mdash; trading extra compute (time) for lower
peak memory. Compare `seg=0` (off) vs `seg=2`.

In [5]:
run_training(make_model("kaiming"), label="checkpoint=OFF", checkpoint_seg=0)
run_training(make_model("kaiming"), label="checkpoint=ON(seg4)", checkpoint_seg=4)
# Expect (GPU): checkpoint ON -> lower peak_mem_MB, slightly higher time_s, same-ish loss.


{'experiment': 'checkpoint=OFF', 'time_s': 7.162, 'peak_mem_MB': 847.8, 'final_loss': 2.3135}
{'experiment': 'checkpoint=ON(seg4)', 'time_s': 9.062, 'peak_mem_MB': 646.5, 'final_loss': 2.318}


{'experiment': 'checkpoint=ON(seg4)',
 'time_s': 9.062,
 'peak_mem_MB': 646.5,
 'final_loss': 2.318}

## 4. Gradient accumulation

Accumulate gradients over several **micro-batches** before one optimizer step to
simulate a larger effective batch without its memory cost. Effective batch is the
same `BATCH` in both runs; `accum=4` processes it as 4 micro-batches of 32.

In [6]:
run_training(make_model("kaiming"), label="accum=1 (batch 8192)", accum=1)
run_training(make_model("kaiming"), label="accum=4 (4x2048)",     accum=4)
# Expect (GPU): accum=4 -> lower peak_mem_MB (smaller live micro-batch), similar loss, a bit slower.


{'experiment': 'accum=1 (batch 8192)', 'time_s': 6.882, 'peak_mem_MB': 847.8, 'final_loss': 2.3149}
{'experiment': 'accum=4 (4x2048)', 'time_s': 7.073, 'peak_mem_MB': 495.6, 'final_loss': 2.3157}


{'experiment': 'accum=4 (4x2048)',
 'time_s': 7.073,
 'peak_mem_MB': 495.6,
 'final_loss': 2.3157}

## 5. Mixed-precision training (AMP)

Run forward/backward in float16/bfloat16 under `autocast` with a `GradScaler` to
prevent underflow. On modern GPUs this cuts memory and speeds up training with
little accuracy loss. (Benefit is GPU-only.)

In [7]:
run_training(make_model("kaiming"), label="precision=FP32", use_amp=False)
run_training(make_model("kaiming"), label="precision=AMP",  use_amp=True)
# Expect (GPU): AMP -> lower peak_mem_MB and lower time_s, comparable loss.


{'experiment': 'precision=FP32', 'time_s': 6.738, 'peak_mem_MB': 847.8, 'final_loss': 2.316}
{'experiment': 'precision=AMP', 'time_s': 2.282, 'peak_mem_MB': 587.8, 'final_loss': 2.3151}


{'experiment': 'precision=AMP',
 'time_s': 2.282,
 'peak_mem_MB': 587.8,
 'final_loss': 2.3151}

## 6. Summary table & findings

In [8]:
summary = pd.DataFrame(results)
display(summary)

# --- auto-computed comparison (works off whatever numbers your run produced) ---
def _row(label):
    r = summary[summary.experiment == label]
    return r.iloc[0] if len(r) else None

def _delta(base, variant, name):
    b, v = _row(base), _row(variant)
    if b is None or v is None:
        print(f"{name}: rows missing"); return
    print(f"{name}:")
    if b.peak_mem_MB and v.peak_mem_MB:
        d = v.peak_mem_MB - b.peak_mem_MB
        print(f"   peak_mem {b.peak_mem_MB:.0f} -> {v.peak_mem_MB:.0f} MB "
              f"({100*d/b.peak_mem_MB:+.0f}%)")
    print(f"   time     {b.time_s:.3f} -> {v.time_s:.3f} s ({v.time_s-b.time_s:+.3f})")
    print(f"   loss     {b.final_loss:.3f} -> {v.final_loss:.3f}")

print()
_delta("checkpoint=OFF", "checkpoint=ON(seg4)", "Activation checkpointing")
_delta("accum=1 (batch 8192)", "accum=4 (4x2048)", "Gradient accumulation")
_delta("precision=FP32", "precision=AMP", "Mixed precision (AMP)")


,experiment,time_s,peak_mem_MB,final_loss
0,init=default,6.930,847.8,2.3029
1,init=kaiming,6.846,847.8,2.3139
2,init=xavier,6.814,847.8,2.3030
3,checkpoint=OFF,7.162,847.8,2.3135
4,checkpoint=ON(seg4),9.062,646.5,2.3180
5,accum=1 (batch 8192),6.882,847.8,2.3149
6,accum=4 (4x2048),7.073,495.6,2.3157
7,precision=FP32,6.738,847.8,2.3160
8,precision=AMP,2.282,587.8,2.3151



Activation checkpointing:
   peak_mem 848 -> 646 MB (-24%)
   time     7.162 -> 9.062 s (+1.900)
   loss     2.313 -> 2.318
Gradient accumulation:
   peak_mem 848 -> 496 MB (-42%)
   time     6.882 -> 7.073 s (+0.191)
   loss     2.315 -> 2.316
Mixed precision (AMP):
   peak_mem 848 -> 588 MB (-31%)
   time     6.738 -> 2.282 s (-4.456)
   loss     2.316 -> 2.315


## Findings

Fill in with your table's numbers; expected trends with this (activation-heavy) config:

- **Tensor creation (CPU vs GPU).** GPU matmul is many times faster than CPU
  (see Section 1) because the large parallel op saturates the GPU.
- **Weight initialisation.** All initialisers converged to ≈ln(10)=2.30 in 20 steps; on this deep plain-MLP task none learned meaningfully within the step budget, so init did not differentiate final loss here. Init's effect is most visible in early-layer gradient scale on trainable architectures.
- **Activation checkpointing.** ON gives **lower peak memory** at the cost of
  **higher time** (activations are recomputed in backward), with ~unchanged loss.
  The saving is visible here only because activations dominate memory; on a small
  model (few narrow layers) checkpointing saves nothing.
- **Gradient accumulation.** `accum=4` runs the same effective batch as four
  micro-batches, so **peak memory drops** (smaller live activation set) while
  **time rises** slightly and loss is ~unchanged. It only helps when a full batch
  would not fit in memory.
- **Mixed precision (AMP).** Half-precision activations **lower peak memory** and,
  on tensor-core-friendly sizes, **reduce time**, with loss comparable to FP32.
  On tiny models the fp16/fp32 bookkeeping can outweigh the saving.

> **If a technique shows no benefit in your run:** the model's activations are
> still small relative to weights. Increase `BATCH` and/or `DEPTH` in the setup
> cell until activations dominate, then checkpointing / accumulation / AMP show
> their expected memory savings. On CPU (`CUDA: False`) the `peak_mem_MB` column
> is `None` and only time/loss are meaningful.
